<a href="https://colab.research.google.com/github/karthikpotnuru7/RAG_ingestion_retrival_pipeline/blob/main/RAG_ingestion_%26_retrival_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

//To upload TXT or PDF files

In [1]:
from google.colab import files

uploaded = files.upload()

Saving Google.txt to Google.txt


//Load Document

In [2]:
!pip install -q langchain-community

from langchain_community.document_loaders import TextLoader

loader = TextLoader("Google.txt")

documents = loader.load()

print("Documents Loaded:", len(documents))
print(documents[0].metadata)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_3762/4186756809.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Documents Loaded: 1
{'source': 'Google.txt'}


//Chunking

In [48]:
# Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the text splitter
text_splitter = RecursiveCharacterTextSplitter(

    # Maximum size of each chunk
    chunk_size=1000,

    # Shared text between neighboring chunks
    chunk_overlap=200
)

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

# Print total number of chunks created
print("Total Chunks Created:", len(chunks))

# Display first chunk
print("\n===== First Chunk =====\n")
print(chunks[0].page_content[:500])

# Display length of first chunk
print("\nLength of First Chunk:", len(chunks[0].page_content))

for i in range(10):
    print(f"Chunk {i+1}: {len(chunks[i].page_content)}")

Total Chunks Created: 298

===== First Chunk =====

﻿Google
Google LLC (/ˈɡuːɡəl/ ⓘ , GOO-gəl) is an Google LLC
American multinational corporation and technology
company focusing on online advertising, search engine
technology, cloud computing, computer software,
quantum computing, e-commerce, consumer
electronics, and artificial intelligence (AI).[9] It has
been referred to as "the most powerful company in the The Google logo used since 2015
world" by the BBC[10] and is one of the world's most
valuable brands.[11][12][13] Google's parent company

Length of First Chunk: 600
Chunk 1: 600
Chunk 2: 867
Chunk 3: 971
Chunk 4: 953
Chunk 5: 730
Chunk 6: 918
Chunk 7: 980
Chunk 8: 877
Chunk 9: 535
Chunk 10: 949


//Connecting API key

In [4]:
!pip install -q openai

In [49]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

//EMBEDDING GENERATION

In [6]:
# Generate embedding for first chunk

response = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=chunks[0].page_content
)

embedding = response.data[0].embedding

print("Embedding Dimension:", len(embedding))

Embedding Dimension: 1536


//GENERATE EMBEDDINGS FOR ALL CHUNKS

In [7]:
# Empty list to store embeddings
embeddings = []

# Loop through every chunk
for i, chunk in enumerate(chunks):

    # Generate embedding for the chunk
    response = client.embeddings.create(
        model="openai/text-embedding-3-small",
        input=chunk.page_content
    )

    # Store embedding vector
    embeddings.append(response.data[0].embedding)

    # Progress update every 25 chunks
    if (i + 1) % 25 == 0:
        print(f"Processed {i+1}/{len(chunks)} chunks")

# Final verification
print("\nTotal Embeddings Generated:", len(embeddings))

Processed 25/298 chunks
Processed 50/298 chunks
Processed 75/298 chunks
Processed 100/298 chunks
Processed 125/298 chunks
Processed 150/298 chunks
Processed 175/298 chunks
Processed 200/298 chunks
Processed 225/298 chunks
Processed 250/298 chunks
Processed 275/298 chunks

Total Embeddings Generated: 298


//VERIFYING

In [8]:
print("Chunks:", len(chunks))
print("Embeddings:", len(embeddings))
print("Dimension:", len(embeddings[0]))

Chunks: 298
Embeddings: 298
Dimension: 1536


// INSTALLING CHROMADB

In [10]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [11]:
import chromadb

print(chromadb.__version__)

1.5.9


//CREATE CHROMADB COLLECTION

In [12]:
import chromadb

# Create ChromaDB client
client_db = chromadb.Client()

# Create a collection (similar to a table in SQL)
collection = client_db.create_collection(
    name="google_documents"
)

print("Collection Created Successfully!")

Collection Created Successfully!


//PREPARE DATA

In [13]:
# Extract plain text from chunks
documents_text = [chunk.page_content for chunk in chunks]

# Create unique IDs for each chunk
ids = [f"chunk_{i}" for i in range(len(chunks))]

print("Documents:", len(documents_text))
print("IDs:", len(ids))
print("Embeddings:", len(embeddings))

Documents: 298
IDs: 298
Embeddings: 298


//STORE DATA IN CHROMADB

In [14]:
collection.add(
    ids=ids,
    documents=documents_text,
    embeddings=embeddings
)

print("Data Stored Successfully!")

Data Stored Successfully!


//RETRIEVAL PIPELINE

//USER QUERY

In [50]:
query = "Who found google and in which year?"

print("User Query:", query)

User Query: Who found google and which in year?


//QUERY EMBEDDING

In [51]:
query_embedding = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=query
).data[0].embedding

print("Query Embedding Dimension:", len(query_embedding))

Query Embedding Dimension: 1536


//SEARCH CHROMADB

In [52]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Retrieved Successfully!")

Retrieved Successfully!


//DISPLAY RETRIEVED CHUNKS

In [53]:
for i, doc in enumerate(results["documents"][0]):

    print(f"\n===== Retrieved Chunk {i+1} =====\n")

    print(doc[:1000])


===== Retrieved Chunk 1 =====

Robin Li in 1996, with Larry Page's PageRank patent experience in HTML, the markup language used
for designing web pages[30]

including a citation to Li's earlier RankDex patent; Li
later went on to create the Chinese search engine
Baidu.[37][38]

Eventually, they changed the name to Google; the name of the search engine was a misspelling of the
word googol,[23][39][40] a very large number written 10100 (1 followed by 100 zeros), picked to signify
that the search engine was intended to provide large quantities of information.[41]

===== Retrieved Chunk 2 =====

242. Koller, David (January 2004). "Origin of the name, "Google." " (https://web.archive.org/web/2
0120627081942/http://graphics.stanford.edu/~dk/google_name_origin.html). Stanford
University. Archived from the original (http://graphics.stanford.edu/~dk/google_name_origin.
html) on June 27, 2012. Retrieved May 28, 2006.

243. Hanley, Rachael. ""From Googol to Google: Co-founder returns" (https://w

//PREPARE RETRIEVED CONTEXT

In [54]:
context = "\n\n".join(results["documents"][0])

print(context[:3000])

Robin Li in 1996, with Larry Page's PageRank patent experience in HTML, the markup language used
for designing web pages[30]

including a citation to Li's earlier RankDex patent; Li
later went on to create the Chinese search engine
Baidu.[37][38]

Eventually, they changed the name to Google; the name of the search engine was a misspelling of the
word googol,[23][39][40] a very large number written 10100 (1 followed by 100 zeros), picked to signify
that the search engine was intended to provide large quantities of information.[41]

242. Koller, David (January 2004). "Origin of the name, "Google." " (https://web.archive.org/web/2
0120627081942/http://graphics.stanford.edu/~dk/google_name_origin.html). Stanford
University. Archived from the original (http://graphics.stanford.edu/~dk/google_name_origin.
html) on June 27, 2012. Retrieved May 28, 2006.

243. Hanley, Rachael. ""From Googol to Google: Co-founder returns" (https://web.archive.org/we
b/20110511111017/http://www.stanforddaily.com

//GENERATE FINAL RAG ANSWER

In [55]:
prompt = f"""
Answer the question using ONLY the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [56]:
response = client.chat.completions.create(
    model="openai/gpt-4o-mini",

    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

Google was co-founded by Larry Page and Sergey Brin in 1998.
